# 03 — Build Profiles and Execution Metrics

This notebook builds historical workload profiles and point-in-time execution features from the canonical Google subset execution and observation datasets.

An occurrence is defined by `recurrence_key + execution_start_time_us`. CPU and memory usage are aggregated across parallel executions within each canonical observation window, then the maximum window total is selected for the occurrence.

Historical features use only prior occurrences from the same `recurrence_key` that have fully completed by the current occurrence's start time. This prevents future information from leaking into pre-execution recommendations.

## Imports and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Find repository root
REPO_ROOT = Path.cwd()

while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
)

EXECUTIONS_PATH = (
    OUTPUT_DIR / "google_subset_executions_v1.parquet"
)

OBSERVATIONS_PATH = (
    OUTPUT_DIR / "google_subset_observations_reconstructed_v1.parquet"
)

# Verify both inputs exist
for path in [EXECUTIONS_PATH, OBSERVATIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook 2 artifact not found:\n{path}\n"
            "Run the updated 02_reconstructExecutions.ipynb first."
        )
# Load reconstructed executions and observations
executions = pd.read_parquet(EXECUTIONS_PATH)
observations = pd.read_parquet(OBSERVATIONS_PATH)


print("Repository root:", REPO_ROOT)

print("\nLoaded Notebook 2 outputs")
print("-------------------------")
print(f"Executions:   {len(executions):,}")
print(f"Observations: {len(observations):,}")

print("\nExecution columns:")
print(executions.columns.tolist())

print("\nObservation columns:")
print(observations.columns.tolist())

executions.head()

Repository root: /Users/sofiasilva/Masters-project/masters-project

Loaded Notebook 2 outputs
-------------------------
Executions:   253,585
Observations: 405,894

Execution columns:
['execution_id', 'collection_id', 'instance_index', 'candidate_instance_key', 'episode_number', 'execution_start_time_us', 'execution_end_time_us', 'observation_count', 'machine_count', 'requested_cpu_ncu', 'requested_memory_normalized', 'average_cpu_ncu', 'average_memory_normalized', 'peak_cpu_ncu', 'peak_memory_normalized', 'runtime_us', 'runtime_sec', 'slack_cpu_ncu', 'slack_memory_normalized', 'user', 'collection_name', 'collection_logical_name', 'user_clean', 'logical_workload_name', 'recurrence_key', 'recurring_job_id']

Observation columns:
['Unnamed: 0', 'time', 'instance_events_type', 'collection_id', 'scheduling_class', 'collection_type', 'priority', 'alloc_collection_id', 'instance_index', 'machine_id', 'resource_request', 'constraint', 'collections_events_type', 'user', 'collection_name', 'col

,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,runtime_sec,slack_cpu_ncu,slack_memory_normalized,user,collection_name,collection_logical_name,user_clean,logical_workload_name,recurrence_key,recurring_job_id
0,100286199||134||episode_1,100286199,134,100286199||134,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
1,100286199||250||episode_1,100286199,250,100286199||250,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
2,100286199||269||episode_1,100286199,269,100286199||269,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
3,100286199||321||episode_1,100286199,321,100286199||321,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
4,100286199||323||episode_1,100286199,323,100286199||323,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...


## Attach recurrence information to reconstructed observations


In [2]:
# Each observation belongs to one reconstructed execution
# Use execution_id to attach its recurrence key and start time

execution_lookup = executions[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
    ]
].copy()

# Validate the execution lookup before merging.
assert execution_lookup["execution_id"].is_unique
assert execution_lookup["execution_id"].notna().all()
assert observations["observation_id"].is_unique

# Join observation-level usage windows with execution metadata
observations = observations.merge(
    execution_lookup,
    on="execution_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Validate the observation-to-execution mapping
print("Observation-to-execution mapping")
print("--------------------------------")
print(f"Observations: {len(observations):,}")
print(f"Matched: {(observations['_merge'] == 'both').sum():,}")
print(f"Unmatched: {(observations['_merge'] != 'both').sum():,}")

assert (observations["_merge"] == "both").all()
assert observations["recurrence_key"].notna().all()
assert observations["execution_start_time_us"].notna().all()

# Remove the temporary merge indicator
observations = observations.drop(columns="_merge")

print("\nObservation-to-execution mapping validation passed.")

observations[
    [
        "observation_id",
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "usage_start_time_us",
        "usage_end_time_us",
        "peak_cpu_ncu",
        "peak_memory_normalized",
    ]
].head()

Observation-to-execution mapping
--------------------------------
Observations: 405,894
Matched: 405,894
Unmatched: 0

Observation-to-execution mapping validation passed.


,observation_id,execution_id,recurrence_key,execution_start_time_us,usage_start_time_us,usage_end_time_us,peak_cpu_ncu,peak_memory_normalized
0,94591244395||144||168846390496||274800000000||...,94591244395||144||episode_1,fn8Ve4Tdl/FVVvwXFGIKe4+Wo4zLjUL/557qdFVYu5M=||...,274800000000,274800000000,275100000000,0.011902,5.935669e-03
1,260697606809||335||85515092||1800713000000||18...,260697606809||335||episode_1,DrrEIEWkWuW7RrZwpHLCN0k0A2J0usJeyt3wtqzZ7Kk=||...,1800713000000,1800713000000,1800714000000,0.000000,9.536743e-07
2,276227177776||376||169321752432||81300000000||...,276227177776||376||episode_1,/ivQBmewiFcXfGJdCUsEKx47NiRE29Tjiq3gw+zR2Cg=||...,81300000000,81300000000,81600000000,0.060059,2.845764e-03
3,10507389885||1977||178294817221||1075500000000...,10507389885||1977||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,1075500000000,1075500000000,1075800000000,0.133301,3.466797e-02
4,25911621841||3907||231364893292||1565315000000...,25911621841||3907||episode_1,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,1565315000000,1565315000000,1565317000000,0.000415,7.629395e-05


## Build occurrence-level CPU and memory peaks


In [3]:
occurrence_keys = [
    "recurrence_key",
    "execution_start_time_us",
]

window_keys = occurrence_keys + [
    "usage_start_time_us",
    "usage_end_time_us",
]

# Check that usage windows are available
assert observations[
    ["usage_start_time_us", "usage_end_time_us"]
].notna().all().all()

# A missing component makes that window's resource total unknown
# min_count=1 prevents entirely missing groups from becoming zero
window_totals = (
    observations
    .groupby(window_keys, dropna=False)
    .agg(
        cpu_total=("peak_cpu_ncu", lambda x:
                   x.sum() if x.notna().all() else np.nan),
        memory_total=("peak_memory_normalized", lambda x:
                      x.sum() if x.notna().all() else np.nan),
    )
    .reset_index()
)

# An occurrence peak is the largest complete window total
# If any required window total is missing, preserve NaN
occurrence_resource_peaks = (
    window_totals
    .groupby(occurrence_keys, dropna=False)
    .agg(
        occurrence_peak_cpu_ncu=(
            "cpu_total",
            lambda x: x.max() if x.notna().all() else np.nan
        ),
        occurrence_peak_memory_normalized=(
            "memory_total",
            lambda x: x.max() if x.notna().all() else np.nan
        ),
    )
    .reset_index()
)

print("Occurrence-level resource aggregation")
print("-------------------------------------")
print(f"Observation rows: {len(observations):,}")
print(f"Usage windows: {len(window_totals):,}")
print(f"Unique occurrences: {len(occurrence_resource_peaks):,}")

print("\nMissing occurrence resource peaks:")
print(
    occurrence_resource_peaks[
        [
            "occurrence_peak_cpu_ncu",
            "occurrence_peak_memory_normalized",
        ]
    ].isna().sum()
)

occurrence_resource_peaks.head()

Occurrence-level resource aggregation
-------------------------------------
Observation rows: 405,894
Usage windows: 4,883
Unique occurrences: 4,879

Missing occurrence resource peaks:
occurrence_peak_cpu_ncu               0
occurrence_peak_memory_normalized    60
dtype: int64


,recurrence_key,execution_start_time_us,occurrence_peak_cpu_ncu,occurrence_peak_memory_normalized
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,0.594543,0.247986
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,0.001840,0.002357
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,0.076355,0.028976
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,0.062744,0.005478
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,0.043335,0.005547


## Validate the known execution occurrences

In [4]:
occurrence_sizes = (
    executions
    .groupby(
        ["recurrence_key", "execution_start_time_us"]
    )
    .size()
    .reset_index(name="execution_count")
)

audited = occurrence_sizes[
    occurrence_sizes["execution_count"] == 6921
]

print("Occurrences with 6,921 executions:", len(audited))

assert len(audited) == 1, (
    "Expected exactly one occurrence with 6,921 executions."
)

audited_peaks = audited.merge(
    occurrence_resource_peaks,
    on=["recurrence_key", "execution_start_time_us"],
    how="left",
    validate="one_to_one",
)

print("\nAudited occurrence resource peaks:")
print(
    audited_peaks[
        [
            "execution_count",
            "occurrence_peak_cpu_ncu",
            "occurrence_peak_memory_normalized",
        ]
    ].to_string(index=False)
)

np.testing.assert_allclose(
    audited_peaks["occurrence_peak_cpu_ncu"],
    [91.877289],
    rtol=1e-5,
)

np.testing.assert_allclose(
    audited_peaks["occurrence_peak_memory_normalized"],
    [0.481828],
    rtol=1e-5,
)

print("\nAudited occurrence validation passed.")


Occurrences with 6,921 executions: 1

Audited occurrence resource peaks:
 execution_count  occurrence_peak_cpu_ncu  occurrence_peak_memory_normalized
            6921                91.877289                           0.481828

Audited occurrence validation passed.


## Build occurrence-level requests and runtime


In [5]:
occurrence_keys = [
    "recurrence_key",
    "execution_start_time_us",
]

def complete_sum(values):
    """Sum only when every component is present."""
    if values.isna().any():
        return np.nan
    return values.sum(min_count=1)

occurrence_execution_metrics = (
    executions
    .groupby(occurrence_keys, as_index=False)
    .agg(
        occurrence_end_time_us=(
            "execution_end_time_us", "max"
        ),
        occurrence_requested_cpu_ncu=(
            "requested_cpu_ncu", complete_sum
        ),
        occurrence_requested_memory_normalized=(
            "requested_memory_normalized", complete_sum
        ),
        execution_row_count=(
            "execution_id", "size"
        ),
    )
)

# Runtime is based on the full occurrence boundaries
occurrence_execution_metrics["occurrence_runtime_sec"] = (
    (
        occurrence_execution_metrics["occurrence_end_time_us"]
        - occurrence_execution_metrics["execution_start_time_us"]
    ) / 1_000_000
)

# Attach corrected resource peaks
occurrences = occurrence_execution_metrics.merge(
    occurrence_resource_peaks,
    on=occurrence_keys,
    how="left",
    validate="one_to_one",
)

# Explicit, stable occurrence identifier
occurrences["occurrence_key"] = (
    occurrences["recurrence_key"].astype("string")
    + "||"
    + occurrences["execution_start_time_us"].astype("string")
)

# Validate occurrence-level records
assert len(occurrences) == len(occurrence_resource_peaks)
assert occurrences["occurrence_key"].notna().all()
assert occurrences["occurrence_key"].is_unique
assert occurrences["execution_row_count"].sum() == len(executions)
assert (occurrences["occurrence_runtime_sec"] > 0).all()

print("Occurrence-level metrics")
print("------------------------")
print(f"Unique occurrences: {len(occurrences):,}")
print(f"Execution rows represented: {occurrences['execution_row_count'].sum():,}")

print("\nMissing values:")
print(
    occurrences[
        [
            "occurrence_requested_cpu_ncu",
            "occurrence_requested_memory_normalized",
            "occurrence_peak_cpu_ncu",
            "occurrence_peak_memory_normalized",
            "occurrence_runtime_sec",
        ]
    ].isna().sum()
)

print("\nOccurrence-level validation passed.")

occurrences.head()


Occurrence-level metrics
------------------------
Unique occurrences: 4,879
Execution rows represented: 253,585

Missing values:
occurrence_requested_cpu_ncu              73
occurrence_requested_memory_normalized    73
occurrence_peak_cpu_ncu                    0
occurrence_peak_memory_normalized         60
occurrence_runtime_sec                     0
dtype: int64

Occurrence-level validation passed.


,recurrence_key,execution_start_time_us,occurrence_end_time_us,occurrence_requested_cpu_ncu,occurrence_requested_memory_normalized,execution_row_count,occurrence_runtime_sec,occurrence_peak_cpu_ncu,occurrence_peak_memory_normalized,occurrence_key
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,1.068726,0.635010,34,300.0,0.594543,0.247986,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,116700000000,0.006077,0.007816,3,300.0,0.001840,0.002357,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,359700000000,0.043762,0.034470,3,300.0,0.076355,0.028976,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,45900000000,0.008194,0.008835,1,300.0,0.062744,0.005478,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,1201200000000,0.008987,0.008820,1,300.0,0.043335,0.005547,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...


## Complete Schema

In [6]:
# Inspect the execution-level dataset
executions.info()

# Preview the columns relevant to Notebook 3
executions[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "execution_end_time_us",
        "runtime_sec",
        "requested_cpu_ncu",
        "requested_memory_normalized",
        "peak_cpu_ncu",
        "peak_memory_normalized",
    ]
].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 253585 entries, 0 to 253584
Data columns (total 26 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   execution_id                 253585 non-null  str    
 1   collection_id                253585 non-null  int64  
 2   instance_index               253585 non-null  int64  
 3   candidate_instance_key       253585 non-null  str    
 4   episode_number               253585 non-null  int64  
 5   execution_start_time_us      253585 non-null  Int64  
 6   execution_end_time_us        253585 non-null  Int64  
 7   observation_count            253585 non-null  int64  
 8   machine_count                253585 non-null  int64  
 9   requested_cpu_ncu            252811 non-null  float64
 10  requested_memory_normalized  252811 non-null  float64
 11  average_cpu_ncu              253585 non-null  float64
 12  average_memory_normalized    253585 non-null  float64
 13  peak_cpu_n

,execution_id,recurrence_key,execution_start_time_us,execution_end_time_us,runtime_sec,requested_cpu_ncu,requested_memory_normalized,peak_cpu_ncu,peak_memory_normalized
0,100286199||134||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,2292300000000,300.0,0.046570,0.028015,0.011154,0.001318
1,100286199||250||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,2292300000000,300.0,0.046570,0.028015,0.011154,0.001318
2,100286199||269||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,2292300000000,300.0,0.046570,0.028015,0.011154,0.001318
3,100286199||321||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,2292300000000,300.0,0.046570,0.028015,0.011154,0.001318
4,100286199||323||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,2292300000000,300.0,0.046570,0.028015,0.011154,0.001318
5,100287787||122||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2248500000000,2248800000000,300.0,0.032410,0.007294,0.010040,0.003429
6,100288420||58||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,976200000000,976500000000,300.0,0.045715,0.028870,0.013321,0.000992
7,10084165805||105||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,959100000000,300.0,0.070435,0.041626,0.015488,0.005989
8,10084165805||112||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,959100000000,300.0,0.070435,0.041626,0.015488,0.005989
9,10084165805||17||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,959100000000,300.0,0.070435,0.041626,0.015488,0.005989


## Validate Canonical Schema

Verify that the execution dataset contains the fields required to construct historical workload profiles and point-in-time features.

In [7]:
# Validate required Notebook 2 columns
required_execution_columns = [
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "execution_end_time_us",
    "runtime_sec",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

required_observation_columns = [
    "observation_id",
    "execution_id",
    "usage_start_time_us",
    "usage_end_time_us",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

missing_execution_columns = (
    set(required_execution_columns) - set(executions.columns)
)

missing_observation_columns = (
    set(required_observation_columns) - set(observations.columns)
)

if missing_execution_columns:
    raise ValueError(
        f"Missing execution columns: {sorted(missing_execution_columns)}"
    )

if missing_observation_columns:
    raise ValueError(
        f"Missing observation columns: {sorted(missing_observation_columns)}"
    )

print("All required Notebook 2 columns are present.")

# Validate execution-level integrity
assert executions["execution_id"].notna().all()
assert executions["execution_id"].is_unique
assert executions["recurrence_key"].notna().all()
assert executions["execution_start_time_us"].notna().all()
assert executions["execution_end_time_us"].notna().all()
assert (executions["runtime_sec"] > 0).all()

print("\nExecution-level integrity checks")
print("--------------------------------")
print(f"Rows:                   {len(executions):,}")
print(f"Unique execution IDs:   {executions['execution_id'].nunique():,}")
print(f"Unique recurrence keys: {executions['recurrence_key'].nunique():,}")
print(f"Duplicate execution IDs: {executions['execution_id'].duplicated().sum():,}")

# Validate observation-level integrity
assert observations["observation_id"].notna().all()
assert observations["observation_id"].is_unique
assert observations["execution_id"].notna().all()
assert observations["usage_start_time_us"].notna().all()
assert observations["usage_end_time_us"].notna().all()
assert (
    observations["usage_end_time_us"]
    > observations["usage_start_time_us"]
).all()

print("\nObservation-level integrity checks")
print("----------------------------------")
print(f"Rows:                   {len(observations):,}")
print(f"Unique observation IDs: {observations['observation_id'].nunique():,}")
print(f"Duplicate observation IDs: {observations['observation_id'].duplicated().sum():,}")

print("\nCanonical schema validation passed.")


All required Notebook 2 columns are present.

Execution-level integrity checks
--------------------------------
Rows:                   253,585
Unique execution IDs:   253,585
Unique recurrence keys: 2,280
Duplicate execution IDs: 0

Observation-level integrity checks
----------------------------------
Rows:                   405,894
Unique observation IDs: 405,894
Duplicate observation IDs: 0

Canonical schema validation passed.


## Order Executions Chronologically

Executions are grouped by `recurrence_key` and ordered by execution start time. Executions sharing the same `recurrence key` and start time belong to the same logical occurrence.

This ordering supports the construction of point-in-time historical features using only prior completed occurrences, preventing data leakage.

In [8]:
history = executions.sort_values(
    by=[
        "recurrence_key",
        "execution_start_time_us",
        "execution_id",
    ]
).reset_index(drop=True)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000


## Build Point-in-Time History
Historical features use only prior occurrences from the same `recurrence_key` that have fully completed before the current occurrence starts.

An occurrence is defined by `recurrence_key` and `execution_start_time_us`. Executions sharing these values belong to the same logical occurrence.
CPU and memory usage are calculated by summing resource usage across parallel executions within each canonical usage window, then taking the maximum window total for the occurrence.

For an occurrence starting at time t, a prior occurrence is eligible for history only when its `occurrence_end_time_us <= t` and it started before t. prior_occurrence_count represents the number of distinct prior completed occurrences.

In [9]:
# Count executions occurring at each workload/start-time point
time_groups = (
    history.groupby(
        ["recurrence_key", "execution_start_time_us"],
        sort=False,
    )
    .size()
    .rename("executions_at_start_time")
    .reset_index()
)

history = history.merge(
    time_groups,
    on=["recurrence_key", "execution_start_time_us"],
    how="left",
    validate="many_to_one",
)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "execution_end_time_us",
        "executions_at_start_time",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us,execution_end_time_us,executions_at_start_time
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34


## Build Historical Resource Features

Historical CPU, memory, and runtime statistics are calculated from prior completed occurrences for each recurring workload.
CPU and memory peaks are calculated by summing usage across parallel executions within each usage window, then selecting the maximum window total per occurrence. Runtime is calculated using the occurrence's start and end times.
Only occurrences that completed before the current occurrence starts are included in its historical profile. Historical Mean, P50, P90, P95, P99, standard deviation, and resource-specific counts are calculated from eligible occurrence-level history.

In [10]:
time_point_summary = (
    history.groupby("recurrence_key")["execution_start_time_us"]
    .nunique()
)

print("Distinct start times per recurrence key")
print("---------------------------------------")
print(time_point_summary.describe())

print(
    "\nTotal recurrence/time groups:",
    history[
        ["recurrence_key", "execution_start_time_us"]
    ].drop_duplicates().shape[0]
)

Distinct start times per recurrence key
---------------------------------------
count    2280.000000
mean        2.139912
std         6.999918
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       274.000000
Name: execution_start_time_us, dtype: float64

Total recurrence/time groups: 4879


In [11]:
# Build point-in-time historical profiles
# Use the corrected occurrence-level metrics constructed earlier
# Do NOT rebuild occurrences from execution-level maxima
occurrences = occurrences.sort_values(
    ["recurrence_key", "execution_start_time_us"]
).reset_index(drop=True)

assert occurrences["occurrence_key"].is_unique
assert len(occurrences) == 4879

# Fields used to calculate historical statistics
profile_fields = {
    "occurrence_peak_cpu_ncu": "hist_peak_cpu",
    "occurrence_peak_memory_normalized": "hist_peak_memory",
    "occurrence_runtime_sec": "hist_runtime",
}

historical_profiles = []

for recurrence_key, group in occurrences.groupby(
    "recurrence_key",
    sort=False,
):
    group = group.sort_values("execution_start_time_us")

    for _, current_occurrence in group.iterrows():
        start_time = current_occurrence["execution_start_time_us"]

        # Only prior, fully completed occurrences are eligible
        prior_occurrences = group[
            (group["execution_start_time_us"] < start_time)
            & (group["occurrence_end_time_us"] <= start_time)
        ]

        record = {
            "occurrence_key": current_occurrence["occurrence_key"],
            "recurrence_key": recurrence_key,
            "execution_start_time_us": start_time,
            "prior_occurrence_count": len(prior_occurrences),
        }

        for field, prefix in profile_fields.items():
            values = prior_occurrences[field].dropna()

            record[f"{prefix}_count"] = len(values)

            if len(values) > 0:
                record[f"{prefix}_mean"] = values.mean()
                record[f"{prefix}_std"] = values.std()
                record[f"{prefix}_p50"] = values.quantile(0.50)
                record[f"{prefix}_p90"] = values.quantile(0.90)
                record[f"{prefix}_p95"] = values.quantile(0.95)
                record[f"{prefix}_p99"] = values.quantile(0.99)
            else:
                record[f"{prefix}_mean"] = np.nan
                record[f"{prefix}_std"] = np.nan
                record[f"{prefix}_p50"] = np.nan
                record[f"{prefix}_p90"] = np.nan
                record[f"{prefix}_p95"] = np.nan
                record[f"{prefix}_p99"] = np.nan

        historical_profiles.append(record)

historical_profiles = pd.DataFrame(historical_profiles)

# Validate historical profile construction
assert len(historical_profiles) == len(occurrences)
assert historical_profiles["occurrence_key"].is_unique
assert historical_profiles["occurrence_key"].notna().all()

print("Historical profile construction")
print("-------------------------------")
print(f"Execution rows: {len(history):,}")
print(f"Unique occurrences: {len(occurrences):,}")
print(f"Historical profiles: {len(historical_profiles):,}")

print("\nHistorical profile construction passed.")

historical_profiles.head()


Historical profile construction
-------------------------------
Execution rows: 253,585
Unique occurrences: 4,879
Historical profiles: 4,879

Historical profile construction passed.


,occurrence_key,recurrence_key,execution_start_time_us,prior_occurrence_count,hist_peak_cpu_count,hist_peak_cpu_mean,hist_peak_cpu_std,hist_peak_cpu_p50,hist_peak_cpu_p90,hist_peak_cpu_p95,...,hist_peak_memory_p90,hist_peak_memory_p95,hist_peak_memory_p99,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p90,hist_runtime_p95,hist_runtime_p99
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,1,1,0.062744,NaN,0.062744,0.062744,0.062744,...,0.005478,0.005478,0.005478,1,300.0,<NA>,300.0,300.0,300.0,300.0


In [12]:
# Validate prior completed occurrence counts
count_checks = []

for _, row in historical_profiles.iterrows():
    recurrence_key = row["recurrence_key"]
    start_time = row["execution_start_time_us"]

    # Independently reconstruct eligible historical occurrences
    expected_prior = occurrences[
        (occurrences["recurrence_key"] == recurrence_key)
        & (occurrences["execution_start_time_us"] < start_time)
        & (occurrences["occurrence_end_time_us"] <= start_time)
    ]

    expected_count = len(expected_prior)

    count_checks.append(
        row["prior_occurrence_count"] == expected_count
    )

# Every occurrence must have the correct historical count
mismatches = len(count_checks) - sum(count_checks)

print("Prior occurrence count mismatches:", mismatches)

assert all(count_checks)
assert (
    historical_profiles["prior_occurrence_count"] >= 0
).all()

# Resource-specific counts cannot exceed eligible history
for resource in ["hist_peak_cpu", "hist_peak_memory", "hist_runtime"]:
    assert (
        historical_profiles[f"{resource}_count"]
        <= historical_profiles["prior_occurrence_count"]
    ).all()

print("Prior occurrence count validation passed.")


Prior occurrence count mismatches: 0
Prior occurrence count validation passed.


## Attach Historical Profiles to Executions

Merge the point-in-time occurrence-level historical profile features back onto each execution.
Executions sharing the same recurrence_key and execution_start_time_us belong to the same occurrence and therefore receive the same historical profile. Only prior occurrences that fully completed by the current occurrence's start time are included in history.

In [13]:
job_metrics = history.merge(
    historical_profiles,
    on=[
        "recurrence_key",
        "execution_start_time_us",
    ],
    how="left",
    validate="many_to_one",
)

print("Historical profile merge validation")
print("-----------------------------------")
print(f"Execution rows: {len(history):,}")
print(f"Job metric rows: {len(job_metrics):,}")
print(f"Columns: {len(job_metrics.columns)}")

print(
    "Duplicate execution IDs:",
    job_metrics["execution_id"].duplicated().sum()
)

print(
    "Missing profile counts:",
    job_metrics["hist_peak_cpu_count"].isna().sum()
)

print(
    "Cold-start executions:",
    (job_metrics["prior_occurrence_count"] == 0).sum()
)

print(
    "Executions with history:",
    (job_metrics["prior_occurrence_count"] > 0).sum()
)

# Validate row integrity
assert len(job_metrics) == len(history)
assert job_metrics["execution_id"].is_unique
assert job_metrics["occurrence_key"].notna().all()
assert job_metrics["hist_peak_cpu_count"].notna().all()
assert job_metrics["prior_occurrence_count"].notna().all()

# Validate identical profiles within each occurrence
profile_columns = [
    "prior_occurrence_count",
] + [
    col for col in historical_profiles.columns
    if col.startswith("hist_")
]

profile_variation = (
    job_metrics
    .groupby("occurrence_key")[profile_columns]
    .nunique(dropna=False)
)

assert (profile_variation <= 1).all().all(), (
    "Executions within an occurrence have inconsistent profiles."
)

print("\nAll parallel executions share identical historical profiles.")
print("Merge validation passed.")

job_metrics.head()



Historical profile merge validation
-----------------------------------
Execution rows: 253,585
Job metric rows: 253,585
Columns: 50
Duplicate execution IDs: 0
Missing profile counts: 0
Cold-start executions: 80801
Executions with history: 172784

All parallel executions share identical historical profiles.
Merge validation passed.


,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,hist_peak_memory_p90,hist_peak_memory_p95,hist_peak_memory_p99,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p90,hist_runtime_p95,hist_runtime_p99
0,83212368472||1008||episode_1,83212368472,1008,83212368472||1008,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,83212368472||1042||episode_1,83212368472,1042,83212368472||1042,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,83212368472||1056||episode_1,83212368472,1056,83212368472||1056,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,83212368472||1096||episode_1,83212368472,1096,83212368472||1096,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
4,83212368472||1099||episode_1,83212368472,1099,83212368472||1099,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN


## Validate Point-in-Time History

Verify that historical profile features are calculated only from prior occurrences that fully completed by the time the current occurrence starts.

This check independently reconstructs eligible occurrence-level history and compares the expected counts and CPU, memory, and runtime statistics against the stored historical features.

In [14]:
# Validate point-in-time historical statistics
profile_fields = {
    "occurrence_peak_cpu_ncu": "hist_peak_cpu",
    "occurrence_peak_memory_normalized": "hist_peak_memory",
    "occurrence_runtime_sec": "hist_runtime",
}

stat_functions = {
    "mean": lambda x: x.mean(),
    "std": lambda x: x.std(),
    "p50": lambda x: x.quantile(0.50),
    "p90": lambda x: x.quantile(0.90),
    "p95": lambda x: x.quantile(0.95),
    "p99": lambda x: x.quantile(0.99),
}

checks = 0

for _, row in historical_profiles.iterrows():

    # Independently identify eligible prior completed occurrences
    prior = occurrences[
        (occurrences["recurrence_key"] == row["recurrence_key"])
        & (
            occurrences["execution_start_time_us"]
            < row["execution_start_time_us"]
        )
        & (
            occurrences["occurrence_end_time_us"]
            <= row["execution_start_time_us"]
        )
    ]

    # Validate distinct prior completed occurrence count
    assert row["prior_occurrence_count"] == len(prior)

    for field, prefix in profile_fields.items():

        # Exclude missing measurements for this resource only
        values = prior[field].dropna()

        # Validate resource-specific historical count
        assert row[f"{prefix}_count"] == len(values)

        # Validate all historical statistics.
        for stat, calculate in stat_functions.items():

            expected = (
                calculate(values)
                if len(values) > 0
                else np.nan
            )

            actual = row[f"{prefix}_{stat}"]

            # Convert pandas nullable values to NumPy-compatible floats
            actual = np.nan if pd.isna(actual) else float(actual)
            expected = np.nan if pd.isna(expected) else float(expected)

            np.testing.assert_allclose(
                actual,
                expected,
                rtol=1e-7,
                atol=1e-10,
                equal_nan=True,
                err_msg=(
                    f"Mismatch for {row['occurrence_key']}: "
                    f"{prefix}_{stat}"
                ),
            )

    checks += 1

print("Point-in-time history validation")
print("--------------------------------")
print(f"Occurrences checked: {checks:,}")
print(f"Historical metrics checked: {checks * 3 * 6:,}")
print("Prior occurrence counts: Passed")
print("Resource-specific counts: Passed")
print("Historical statistics: Passed")
print("Missing-value handling: Passed")

print("\nPoint-in-time history validation passed.")



Point-in-time history validation
--------------------------------
Occurrences checked: 4,879
Historical metrics checked: 87,822
Prior occurrence counts: Passed
Resource-specific counts: Passed
Historical statistics: Passed
Missing-value handling: Passed

Point-in-time history validation passed.


## Validate Cold Starts and Missing Values

Cold-start occurrences have no prior completed occurrence history, so historical statistics should remain unavailable rather than being imputed.

Missing source measurements are also preserved. Resource-specific historical counts may therefore be smaller than the total number of prior occurrences.

In [15]:
cold_start = job_metrics[
    job_metrics["prior_occurrence_count"] == 0
]

history_available = job_metrics[
    job_metrics["prior_occurrence_count"] > 0
]

resource_prefixes = [
    "hist_peak_cpu",
    "hist_peak_memory",
    "hist_runtime",
]

stat_names = [
    "mean", "std", "p50", "p90", "p95", "p99"
]

historical_stat_columns = [
    f"{prefix}_{stat}"
    for prefix in resource_prefixes
    for stat in stat_names
]

print("Cold-start validation")
print("---------------------")
print(f"Cold-start execution rows: {len(cold_start):,}")

# Cold starts must have no historical statistics
assert cold_start[
    historical_stat_columns
].isna().all().all()

# Cold starts must have zero resource-specific counts
for prefix in resource_prefixes:
    assert (cold_start[f"{prefix}_count"] == 0).all()

print("Cold starts with non-null historical stats: 0")

# Validate historical resource availability
print("\nExecution rows with history")
print("---------------------------")
print(f"Executions with prior history: {len(history_available):,}")

for prefix in resource_prefixes:

    count_col = f"{prefix}_count"
    mean_col = f"{prefix}_mean"
    std_col = f"{prefix}_std"

    # Counts cannot exceed eligible prior occurrences
    assert (
        job_metrics[count_col]
        <= job_metrics["prior_occurrence_count"]
    ).all()

    # No valid historical measurements means no statistics
    no_data = job_metrics[count_col] == 0

    assert job_metrics.loc[
        no_data, historical_stat_columns[
            resource_prefixes.index(prefix) * 6:
            (resource_prefixes.index(prefix) + 1) * 6
        ]
    ].isna().all().all()

    # At least one valid measurement produces a mean
    has_data = job_metrics[count_col] > 0

    assert job_metrics.loc[
        has_data, mean_col
    ].notna().all()

    # Sample standard deviation requires at least two values
    insufficient_data = job_metrics[count_col] < 2

    assert job_metrics.loc[
        insufficient_data, std_col
    ].isna().all()

print("\nHistorical feature missing values:")
print(
    history_available[
        historical_stat_columns
    ].isna().sum()
)

print("\nCold-start and missing-value validation passed.")


Cold-start validation
---------------------
Cold-start execution rows: 80,801
Cold starts with non-null historical stats: 0

Execution rows with history
---------------------------
Executions with prior history: 172,784

Historical feature missing values:
hist_peak_cpu_mean           0
hist_peak_cpu_std        46740
hist_peak_cpu_p50            0
hist_peak_cpu_p90            0
hist_peak_cpu_p95            0
hist_peak_cpu_p99            0
hist_peak_memory_mean     2357
hist_peak_memory_std     48580
hist_peak_memory_p50      2357
hist_peak_memory_p90      2357
hist_peak_memory_p95      2357
hist_peak_memory_p99      2357
hist_runtime_mean            0
hist_runtime_std         46740
hist_runtime_p50             0
hist_runtime_p90             0
hist_runtime_p95             0
hist_runtime_p99             0
dtype: int64

Cold-start and missing-value validation passed.


## Independent occurrence-level validation
Independently check the underlying occurrence-level resource totals and runtime.

In [16]:
# Independently validate occurrence-level metrics
occurrence_keys = [
    "recurrence_key",
    "execution_start_time_us",
]

window_keys = occurrence_keys + [
    "usage_start_time_us",
    "usage_end_time_us",
]

# Recalculate window totals directly from observations
independent_windows = (
    observations
    .groupby(window_keys, dropna=False)
    .agg(
        cpu_sum=("peak_cpu_ncu", "sum"),
        memory_sum=("peak_memory_normalized", "sum"),
        cpu_count=("peak_cpu_ncu", "count"),
        memory_count=("peak_memory_normalized", "count"),
        observation_count=("observation_id", "size"),
    )
    .reset_index()
)

# Invalidate totals if any component was missing
independent_windows.loc[
    independent_windows["cpu_count"]
    != independent_windows["observation_count"],
    "cpu_sum",
] = np.nan

independent_windows.loc[
    independent_windows["memory_count"]
    != independent_windows["observation_count"],
    "memory_sum",
] = np.nan

# Recalculate occurrence peaks
independent_peaks = (
    independent_windows
    .groupby(occurrence_keys, dropna=False)
    .agg(
        expected_cpu_peak=("cpu_sum", "max"),
        expected_memory_peak=("memory_sum", "max"),
        valid_cpu_windows=("cpu_sum", "count"),
        valid_memory_windows=("memory_sum", "count"),
        total_windows=("cpu_sum", "size"),
    )
    .reset_index()
)

# If any window is incomplete, invalidate the occurrence peak
independent_peaks.loc[
    independent_peaks["valid_cpu_windows"]
    != independent_peaks["total_windows"],
    "expected_cpu_peak",
] = np.nan

independent_peaks.loc[
    independent_peaks["valid_memory_windows"]
    != independent_peaks["total_windows"],
    "expected_memory_peak",
] = np.nan

# Independently reconstruct occurrence runtime boundaries
independent_runtime = (
    executions
    .groupby(occurrence_keys, as_index=False)
    .agg(
        expected_end_time_us=("execution_end_time_us", "max"),
    )
)

independent_runtime["expected_runtime_sec"] = (
    independent_runtime["expected_end_time_us"]
    - independent_runtime["execution_start_time_us"]
) / 1_000_000

# Compare independent calculations with saved occurrence metrics
validation = (
    occurrences
    .merge(
        independent_peaks,
        on=occurrence_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        independent_runtime,
        on=occurrence_keys,
        how="left",
        validate="one_to_one",
    )
)

assert len(validation) == len(occurrences)

for actual, expected in [
    ("occurrence_peak_cpu_ncu", "expected_cpu_peak"),
    ("occurrence_peak_memory_normalized", "expected_memory_peak"),
    ("occurrence_runtime_sec", "expected_runtime_sec"),
]:
    np.testing.assert_allclose(
        validation[actual].to_numpy(dtype=float, na_value=np.nan),
        validation[expected].to_numpy(dtype=float, na_value=np.nan),
        rtol=1e-7,
        atol=1e-10,
        equal_nan=True,
    )

print("Independent occurrence-level validation")
print("---------------------------------------")
print(f"Occurrences checked: {len(validation):,}")
print("CPU peaks: Passed")
print("Memory peaks: Passed")
print("Runtime boundaries: Passed")
print("\nIndependent occurrence-level validation passed.")


Independent occurrence-level validation
---------------------------------------
Occurrences checked: 4,879
CPU peaks: Passed
Memory peaks: Passed
Runtime boundaries: Passed

Independent occurrence-level validation passed.


## Validate Requested CPU and Memory
Requested resources to be aggregated across all executions belonging to an occurrence, while preserving missing values.

In [17]:
# Independently validate occurrence-level requested resources
occurrence_keys = [
    "recurrence_key",
    "execution_start_time_us",
]

# Recalculate requested resource totals from execution records
request_validation = (
    executions
    .groupby(occurrence_keys, dropna=False)
    .agg(
        expected_requested_cpu=(
            "requested_cpu_ncu",
            lambda x: x.sum(min_count=1)
            if x.notna().all() else np.nan
        ),
        expected_requested_memory=(
            "requested_memory_normalized",
            lambda x: x.sum(min_count=1)
            if x.notna().all() else np.nan
        ),
        expected_execution_count=("execution_id", "size"),
    )
    .reset_index()
)

# Compare against occurrence-level metrics
validation = occurrences.merge(
    request_validation,
    on=occurrence_keys,
    how="left",
    validate="one_to_one",
)

assert len(validation) == len(occurrences)

np.testing.assert_allclose(
    validation["occurrence_requested_cpu_ncu"].to_numpy(
        dtype=float, na_value=np.nan
    ),
    validation["expected_requested_cpu"].to_numpy(
        dtype=float, na_value=np.nan
    ),
    rtol=1e-7,
    atol=1e-10,
    equal_nan=True,
)

np.testing.assert_allclose(
    validation["occurrence_requested_memory_normalized"].to_numpy(
        dtype=float, na_value=np.nan
    ),
    validation["expected_requested_memory"].to_numpy(
        dtype=float, na_value=np.nan
    ),
    rtol=1e-7,
    atol=1e-10,
    equal_nan=True,
)

assert (
    validation["execution_row_count"]
    == validation["expected_execution_count"]
).all()

print("Requested resource validation")
print("-----------------------------")
print(f"Occurrences checked: {len(validation):,}")
print("Requested CPU totals: Passed")
print("Requested memory totals: Passed")
print("Execution counts: Passed")

print(
    "Missing requested CPU occurrences:",
    validation["occurrence_requested_cpu_ncu"].isna().sum()
)
print(
    "Missing requested memory occurrences:",
    validation["occurrence_requested_memory_normalized"].isna().sum()
)

print("\nRequested resource validation passed.")


Requested resource validation
-----------------------------
Occurrences checked: 4,879
Requested CPU totals: Passed
Requested memory totals: Passed
Execution counts: Passed
Missing requested CPU occurrences: 73
Missing requested memory occurrences: 73

Requested resource validation passed.


### Target Ownership

This stage creates point-in-time historical profile features and preserves each execution's observed outcomes, including `peak_cpu_ncu` and `peak_memory_normalized`.

It does not create a shifted next-execution prediction target. Any target shifting required for downstream modeling is created in the downstream splitting and modeling stage after chronological ordering and leakage controls are established.

## Save Stage Outputs

Save the point-in-time recurrence profiles and execution-level metrics for downstream splitting, baseline generation, and recommendation evaluation.

These artifacts preserve the canonical execution schema and historical features without applying a cold-start threshold or train/validation/test split.

In [18]:
PROFILE_OUTPUT_PATH = (
    OUTPUT_DIR / "job_profiles_google_subset_history_v1.parquet"
)

METRICS_OUTPUT_PATH = (
    OUTPUT_DIR / "job_metrics_v0.parquet"
)

# Combine occurrence-level metrics with historical profiles
occurrence_profiles = occurrences.merge(
    historical_profiles,
    on=[
        "occurrence_key",
        "recurrence_key",
        "execution_start_time_us",
    ],
    how="left",
    validate="one_to_one",
)

# Validate occurrence-level output
assert len(occurrence_profiles) == len(occurrences)
assert occurrence_profiles["occurrence_key"].is_unique
assert occurrence_profiles["occurrence_key"].notna().all()
assert occurrence_profiles["prior_occurrence_count"].notna().all()

# Validate execution-level output
assert len(job_metrics) == len(executions)
assert job_metrics["execution_id"].is_unique
assert job_metrics["occurrence_key"].notna().all()

# Save occurrence-level profiles
occurrence_profiles.to_parquet(
    PROFILE_OUTPUT_PATH,
    index=False,
)

# Save execution-level metrics
job_metrics.to_parquet(
    METRICS_OUTPUT_PATH,
    index=False,
)

print("Saved stage outputs")
print("-------------------")
print(f"Occurrence profiles: {PROFILE_OUTPUT_PATH}")
print(f"Execution metrics:   {METRICS_OUTPUT_PATH}")

print("\nSaved row counts")
print("----------------")
print(f"Occurrence profiles: {len(occurrence_profiles):,}")
print(f"Execution metrics:   {len(job_metrics):,}")

print("\nStage outputs saved successfully.")


Saved stage outputs
-------------------
Occurrence profiles: /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/job_profiles_google_subset_history_v1.parquet
Execution metrics:   /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/job_metrics_v0.parquet

Saved row counts
----------------
Occurrence profiles: 4,879
Execution metrics:   253,585

Stage outputs saved successfully.


## Validate Saved Outputs

Reload the saved artifacts and verify their row counts, identifiers, and execution-level integrity.

In [19]:
saved_profiles = pd.read_parquet(PROFILE_OUTPUT_PATH)
saved_metrics = pd.read_parquet(METRICS_OUTPUT_PATH)

print("Saved artifact validation")
print("-------------------------")

# Validate row counts
assert len(saved_profiles) == len(occurrence_profiles)
assert len(saved_metrics) == len(job_metrics)

# Validate occurrence-level identifiers
assert saved_profiles["occurrence_key"].notna().all()
assert saved_profiles["occurrence_key"].is_unique

# Validate execution-level identifiers
assert saved_metrics["execution_id"].notna().all()
assert saved_metrics["execution_id"].is_unique
assert saved_metrics["occurrence_key"].notna().all()

# Every execution must belong to a saved occurrence
assert set(saved_metrics["occurrence_key"]) == set(
    saved_profiles["occurrence_key"]
)

# Validate saved values and missingness
def normalize_for_comparison(df):
    result = df.copy().reset_index(drop=True)

    for col in result.columns:
        if col.startswith("hist_") or col.startswith("occurrence_"):
            if col != "occurrence_key":
                result[col] = pd.to_numeric(
                    result[col],
                    errors="raise"
                ).astype("float64")

    return result


pd.testing.assert_frame_equal(
    normalize_for_comparison(saved_profiles),
    normalize_for_comparison(occurrence_profiles),
    check_dtype=False,
    check_exact=False,
    rtol=1e-7,
    atol=1e-10,
)

pd.testing.assert_frame_equal(
    normalize_for_comparison(saved_metrics),
    normalize_for_comparison(job_metrics),
    check_dtype=False,
    check_exact=False,
    rtol=1e-7,
    atol=1e-10,
)

# Validate runtime integrity
assert (saved_profiles["occurrence_runtime_sec"] > 0).all()
assert (saved_metrics["runtime_sec"] > 0).all()

# Validation summary
print(f"Occurrence profiles: {len(saved_profiles):,}")
print(f"Execution metrics:   {len(saved_metrics):,}")

print(
    "Unique occurrence IDs:",
    f"{saved_profiles['occurrence_key'].nunique():,}"
)

print(
    "Unique execution IDs:",
    f"{saved_metrics['execution_id'].nunique():,}"
)

print("\nIdentifier integrity: Passed")
print("Saved values and missingness: Passed")
print("Runtime integrity: Passed")

print("\nSaved artifact validation passed.")


Saved artifact validation
-------------------------
Occurrence profiles: 4,879
Execution metrics:   253,585
Unique occurrence IDs: 4,879
Unique execution IDs: 253,585

Identifier integrity: Passed
Saved values and missingness: Passed
Runtime integrity: Passed

Saved artifact validation passed.



## Summary

This notebook:

- Loaded the canonical reconstructed execution and observation datasets.
- Grouped recurring workloads using `recurrence_key`.
- Defined occurrences using `recurrence_key + execution_start_time_us`.
- Aggregated CPU and memory usage across parallel executions per observation window, then selected the maximum window total per occurrence.
- Aggregated requested CPU and memory across executions while preserving missing values.
- Calculated occurrence runtime using execution start and end boundaries.
- Built point-in-time historical profiles using only prior completed occurrences.
- Calculated historical Mean, standard deviation, P50, P90, P95, P99, and resource-specific occurrence counts.
- Preserved cold starts and missing source measurements without imputation.
- Attached historical profiles to execution-level records.
- Validated occurrence metrics, historical statistics, cold starts, missing values, and execution integrity.
- Saved occurrence-level profiles and execution-level metrics for downstream pipeline stages.